# Reshape and missing values

这份 notebook 聚焦整形和缺失值处理，并把纯 pandas 写法和 tidypy 写法放在一起比较。


## 这份示例包含什么

1. 宽表转长表
2. 长表转宽表
3. 拆列和合列
4. 缺失值处理


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from pandas.testing import assert_frame_equal


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    return start


ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

from tidypy.tidy import (
    drop_na,
    fill_na,
    pivot_longer,
    pivot_wider,
    replace_na,
    separate,
    starts_with,
    unite,
)


## 1. 准备一份宽表


In [2]:
df = pd.DataFrame(
    {
        "id": [1, 2, 3],
        "dept": ["Sales", "Tech", "HR"],
        "score_math": [90.0, None, 84.0],
        "score_eng": [85.0, 92.0, None],
        "code": ["sales-sh", "tech-bj", "hr-sh"],
    }
)

df


,id,dept,score_math,score_eng,code
0,1,Sales,90.0,85.0,sales-sh
1,2,Tech,NaN,92.0,tech-bj
2,3,HR,84.0,NaN,hr-sh


## 2. 对比宽表转长表


In [3]:
pandas_long = df.melt(
    id_vars=["id", "dept", "code"],
    value_vars=["score_math", "score_eng"],
    var_name="metric",
    value_name="value",
)

pandas_long


,id,dept,code,metric,value
0,1,Sales,sales-sh,score_math,90.0
1,2,Tech,tech-bj,score_math,NaN
2,3,HR,hr-sh,score_math,84.0
3,1,Sales,sales-sh,score_eng,85.0
4,2,Tech,tech-bj,score_eng,92.0
5,3,HR,hr-sh,score_eng,NaN


In [4]:
tidypy_long = pivot_longer(
    df,
    starts_with("score_"),
    names_to="metric",
    values_to="value",
)

tidypy_long


,id,dept,code,metric,value
0,1,Sales,sales-sh,score_math,90.0
1,2,Tech,tech-bj,score_math,NaN
2,3,HR,hr-sh,score_math,84.0
3,1,Sales,sales-sh,score_eng,85.0
4,2,Tech,tech-bj,score_eng,92.0
5,3,HR,hr-sh,score_eng,NaN


In [5]:
assert_frame_equal(pandas_long, tidypy_long)
print("Wide-to-long matches.")


Wide-to-long matches.


## 3. 对比长表转宽表


In [6]:
pandas_wide = (
    pandas_long.pivot_table(
        index=["id", "dept", "code"],
        columns="metric",
        values="value",
        aggfunc="first",
        sort=False,
    )
    .reset_index()
    .rename_axis(None, axis=1)
)

pandas_wide


,id,dept,code,score_math,score_eng
0,1,Sales,sales-sh,90.0,85.0
1,2,Tech,tech-bj,NaN,92.0
2,3,HR,hr-sh,84.0,NaN


In [7]:
tidypy_wide = pivot_wider(
    tidypy_long,
    ["id", "dept", "code"],
    "metric",
    "value",
)

tidypy_wide


,id,dept,code,score_math,score_eng
0,1,Sales,sales-sh,90.0,85.0
1,2,Tech,tech-bj,NaN,92.0
2,3,HR,hr-sh,84.0,NaN


In [8]:
assert_frame_equal(pandas_wide, tidypy_wide)
print("Long-to-wide matches.")


Long-to-wide matches.


## 4. 对比拆列和合列


In [9]:
pandas_parts = (
    df.drop(columns=["code"])
      .join(df["code"].str.split("-", expand=True).set_axis(["team", "city"], axis=1))
)

pandas_parts


,id,dept,score_math,score_eng,team,city
0,1,Sales,90.0,85.0,sales,sh
1,2,Tech,NaN,92.0,tech,bj
2,3,HR,84.0,NaN,hr,sh


In [10]:
tidypy_parts = separate(
    df,
    "code",
    ["team", "city"],
    sep="-",
)

tidypy_parts


,id,dept,score_math,score_eng,team,city
0,1,Sales,90.0,85.0,sales,sh
1,2,Tech,NaN,92.0,tech,bj
2,3,HR,84.0,NaN,hr,sh


In [11]:
assert_frame_equal(pandas_parts, tidypy_parts)
print("Separate matches.")


Separate matches.


In [12]:
pandas_united = pandas_parts.assign(
    code_again=lambda x: x[["team", "city"]].astype(str).agg("-".join, axis=1)
)

pandas_united


,id,dept,score_math,score_eng,team,city,code_again
0,1,Sales,90.0,85.0,sales,sh,sales-sh
1,2,Tech,NaN,92.0,tech,bj,tech-bj
2,3,HR,84.0,NaN,hr,sh,hr-sh


In [13]:
tidypy_united = unite(
    tidypy_parts,
    "code_again",
    "team",
    "city",
    sep="-",
    remove=False,
)

tidypy_united


,id,dept,score_math,score_eng,team,city,code_again
0,1,Sales,90.0,85.0,sales,sh,sales-sh
1,2,Tech,NaN,92.0,tech,bj,tech-bj
2,3,HR,84.0,NaN,hr,sh,hr-sh


In [14]:
assert_frame_equal(pandas_united, tidypy_united)
print("Unite matches.")


Unite matches.


## 5. 对比缺失值处理


In [15]:
pandas_drop = df.dropna(subset=["score_math"])
pandas_fill = df.assign(score_math=df["score_math"].ffill())
pandas_replace = df.fillna({"score_math": 0, "score_eng": 0})

{"dropna": pandas_drop, "ffill": pandas_fill, "fillna": pandas_replace}


{'dropna':    id   dept  score_math  score_eng      code
 0   1  Sales        90.0       85.0  sales-sh
 2   3     HR        84.0        NaN     hr-sh,
 'ffill':    id   dept  score_math  score_eng      code
 0   1  Sales        90.0       85.0  sales-sh
 1   2   Tech        90.0       92.0   tech-bj
 2   3     HR        84.0        NaN     hr-sh,
 'fillna':    id   dept  score_math  score_eng      code
 0   1  Sales        90.0       85.0  sales-sh
 1   2   Tech         0.0       92.0   tech-bj
 2   3     HR        84.0        0.0     hr-sh}

In [16]:
tidypy_drop = drop_na(
    df,
    "score_math",
)
tidypy_fill = fill_na(
    df,
    "score_math",
    direction="down",
)
tidypy_replace = replace_na(
    df,
    {"score_math": 0, "score_eng": 0},
)

{"drop_na": tidypy_drop, "fill_na": tidypy_fill, "replace_na": tidypy_replace}


{'drop_na':    id   dept  score_math  score_eng      code
 0   1  Sales        90.0       85.0  sales-sh
 2   3     HR        84.0        NaN     hr-sh,
 'fill_na':    id   dept  score_math  score_eng      code
 0   1  Sales        90.0       85.0  sales-sh
 1   2   Tech        90.0       92.0   tech-bj
 2   3     HR        84.0        NaN     hr-sh,
 'replace_na':    id   dept  score_math  score_eng      code
 0   1  Sales        90.0       85.0  sales-sh
 1   2   Tech         0.0       92.0   tech-bj
 2   3     HR        84.0        0.0     hr-sh}

In [17]:
assert_frame_equal(pandas_drop, tidypy_drop)
assert_frame_equal(pandas_fill, tidypy_fill)
assert_frame_equal(pandas_replace, tidypy_replace)
print("Missing-value helpers match.")


Missing-value helpers match.


## 练习

新增一个 `score_logic` 列，然后重跑 `pivot_longer()` 对比。观察 pandas 写法和 selector 写法哪个改动更小。
